In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table("bank_pulse.silver_financials")

bank_window = Window.partitionBy("cert").orderBy("report_date")

gold = (silver

    .withColumn("quarter_num", F.quarter(F.col("report_date")))

    .withColumn("net_interest_income",
        F.col("interest_income") - F.col("interest_expense"))

    .withColumn("nim",
        F.when(F.col("total_assets") > 0,
            (F.col("interest_income") - F.col("interest_expense"))
            / F.col("total_assets") * 100
            * (4.0 / F.col("quarter_num"))
        ).otherwise(None))

    .withColumn("npl_ratio", F.col("nonperforming_assets"))

    .withColumn("peer_group",
        F.when(F.col("total_assets") < 100000,    F.lit("Community (<$100M)"))
         .when(F.col("total_assets") < 1000000,   F.lit("Mid-Size ($100M-$1B)"))
         .when(F.col("total_assets") < 10000000,  F.lit("Regional ($1B-$10B)"))
         .when(F.col("total_assets") < 100000000, F.lit("Large ($10B-$100B)"))
         .otherwise(                               F.lit("Mega (>$100B)")))

    .withColumn("peer_group_sort",
        F.when(F.col("total_assets") < 100000,    F.lit(1))
         .when(F.col("total_assets") < 1000000,   F.lit(2))
         .when(F.col("total_assets") < 10000000,  F.lit(3))
         .when(F.col("total_assets") < 100000000, F.lit(4))
         .otherwise(                               F.lit(5)))

    .withColumn("assets_prior_qtr",    F.lag("total_assets", 1).over(bank_window))
    .withColumn("nim_prior_qtr",       F.lag("nim", 1).over(bank_window))
    .withColumn("npl_ratio_prior_qtr", F.lag("npl_ratio", 1).over(bank_window))
    .withColumn("roa_prior_qtr",       F.lag("roa", 1).over(bank_window))

    .withColumn("assets_qoq_chg",
        F.when(F.col("assets_prior_qtr") > 0,
            (F.col("total_assets") - F.col("assets_prior_qtr"))
            / F.col("assets_prior_qtr") * 100
        ).otherwise(None))

    .withColumn("nim_qoq_chg",
        F.col("nim") - F.col("nim_prior_qtr"))

    .withColumn("npl_ratio_qoq_chg",
        F.col("npl_ratio") - F.col("npl_ratio_prior_qtr"))

    .withColumn("roa_qoq_chg",
        F.col("roa") - F.col("roa_prior_qtr"))

    .drop("assets_prior_qtr", "nim_prior_qtr", "npl_ratio_prior_qtr", "roa_prior_qtr",
          "quarter_num", "dq_missing_assets", "dq_negative_assets",
          "dq_missing_capital_ratio", "dq_missing_report_date")
)

(gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("report_date")
    .saveAsTable("bank_pulse.gold_bank_metrics")
)

print("✓ Gold written")

display(spark.sql("""
    SELECT report_date, COUNT(DISTINCT cert) AS bank_count,
        ROUND(AVG(nim), 4)       AS avg_nim,
        ROUND(AVG(npl_ratio), 4) AS avg_npl_ratio,
        ROUND(AVG(roa), 4)       AS avg_roa
    FROM bank_pulse.gold_bank_metrics
    WHERE nim IS NOT NULL
    GROUP BY report_date
    ORDER BY report_date DESC
    LIMIT 5
"""))

In [0]:
spark.sql("SHOW DATABASES").show()

In [0]:
spark.sql("SHOW TABLES IN bank_pulse").show()